# GDPR & Data Protection — Exploratory Data Analysis
**Author:** Francisco Costa  
**Dataset:** GDPR DataProtection SQLite Database  
**Period:** January 2022 – December 2024  
**Purpose:** Pre-dashboard exploratory analysis to identify key patterns, compliance gaps, and risk concentrations before building the Power BI report.

---


## 1. Environment Setup & Data Loading

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

NAVY   = '#1A237E'
GREEN  = '#1B5E20'
RED    = '#B71C1C'
AMBER  = '#E65100'
LIGHT  = '#EEF2F7'

# ── Load all tables ────────────────────────────────────────────────────────
DB_PATH = 'gdpr_dataprotection.db'
conn = sqlite3.connect(DB_PATH)

breaches    = pd.read_sql('SELECT * FROM Breaches',               conn)
requests    = pd.read_sql('SELECT * FROM Subject_Requests',       conn)
activities  = pd.read_sql('SELECT * FROM Processing_Activities',  conn)
actions     = pd.read_sql('SELECT * FROM Regulatory_Actions',     conn)
findings    = pd.read_sql('SELECT * FROM Audit_Findings',         conn)
calendar    = pd.read_sql('SELECT * FROM Calendar',               conn)

conn.close()

# ── Parse dates ────────────────────────────────────────────────────────────
breaches['DetectionDate']   = pd.to_datetime(breaches['DetectionDate'])
requests['RequestDate']     = pd.to_datetime(requests['RequestDate'])
findings['AuditDate']       = pd.to_datetime(findings['AuditDate'])
actions['IssueDate']        = pd.to_datetime(actions['IssueDate'])

breaches['Year']  = breaches['DetectionDate'].dt.year
breaches['Month'] = breaches['DetectionDate'].dt.month
breaches['YM']    = breaches['DetectionDate'].dt.to_period('M')

requests['Year']  = requests['RequestDate'].dt.year
requests['Month'] = requests['RequestDate'].dt.month

findings['Year']  = findings['AuditDate'].dt.year
actions['Year']   = actions['IssueDate'].dt.year

print('Data loaded successfully')
print(f'  Breaches:              {len(breaches):>5} rows')
print(f'  Subject Requests:      {len(requests):>5} rows')
print(f'  Processing Activities: {len(activities):>5} rows')
print(f'  Regulatory Actions:    {len(actions):>5} rows')
print(f'  Audit Findings:        {len(findings):>5} rows')


## 2. Dataset Overview

In [ ]:
print('=== BREACHES TABLE ===')
print(breaches.dtypes)
print()
print(breaches.describe(include='all').T[['count','unique','top','mean','min','max']].to_string())


In [ ]:
print('=== NULL CHECK — ALL TABLES ===')
for name, df in [('Breaches', breaches), ('Requests', requests),
                 ('Activities', activities), ('Actions', actions), ('Findings', findings)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f'\n{name}:')
    if len(nulls) == 0:
        print('  No nulls — clean dataset')
    else:
        for col, n in nulls.items():
            print(f'  {col}: {n} nulls ({n/len(df)*100:.1f}%)')


## 3. Breach Analysis (Article 33 GDPR)

**Legal context:** Article 33 GDPR requires organisations to notify the supervisory authority of a personal data breach **within 72 hours** of becoming aware of it, unless the breach is unlikely to result in a risk to individuals. Failure to notify on time is itself a separate infringement.

**Questions this section answers:**
- What is our overall 72-hour notification compliance rate?
- Which breach types are most frequent and most severe?
- Which departments generate the highest risk?
- How has breach volume and compliance trended over 3 years?


In [ ]:
# ── 3.1 Overall compliance rate ───────────────────────────────────────────
total = len(breaches)
on_time = breaches['Within72Hours'].sum()
late = total - on_time
rate = on_time / total * 100

print(f'Total Breaches:          {total}')
print(f'Notified Within 72hrs:   {on_time} ({rate:.1f}%)')
print(f'Notified Late:           {late} ({100-rate:.1f}%)')
print(f'Average Notif. Hours:    {breaches["NotificationHours"].mean():.1f}')
print(f'Max Notif. Hours:        {breaches["NotificationHours"].max():.0f}')
print()
print('ASSESSMENT:', end=' ')
if rate >= 90: print('COMPLIANT (>=90%)')
elif rate >= 75: print('AT RISK (75-89%)')
else: print('NON-COMPLIANT (<75%)')


In [ ]:
# ── 3.2 Compliance by severity ────────────────────────────────────────────
sev_order = ['Critical', 'High', 'Medium', 'Low']
sev = breaches.groupby('Severity').agg(
    Total=('BreachID','count'),
    OnTime=('Within72Hours','sum'),
    AvgHours=('NotificationHours','mean'),
    TotalSubjects=('SubjectsAffected','sum')
).reindex(sev_order)
sev['Late'] = sev['Total'] - sev['OnTime']
sev['ComplianceRate'] = (sev['OnTime'] / sev['Total'] * 100).round(1)
print(sev[['Total','OnTime','Late','ComplianceRate','AvgHours','TotalSubjects']].to_string())


In [ ]:
# ── 3.3 Severity compliance chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar — on time vs late by severity
ax = axes[0]
sev_plot = sev[['OnTime','Late']].reindex(sev_order)
bars_on = ax.barh(sev_order, sev_plot['OnTime'], color=GREEN, label='On Time')
bars_late = ax.barh(sev_order, sev_plot['Late'], left=sev_plot['OnTime'], color=RED, label='Late')
for i, (v, l) in enumerate(zip(sev_plot['OnTime'], sev_plot['Late'])):
    ax.text(v/2, i, str(v), ha='center', va='center', color='white', fontsize=10, fontweight='bold')
    ax.text(v + l/2, i, str(l), ha='center', va='center', color='white', fontsize=10, fontweight='bold')
ax.set_title('72hr Compliance by Severity')
ax.legend()
ax.set_xlabel('Number of Breaches')

# Compliance rate by severity
ax2 = axes[1]
colours = [RED if r < 75 else AMBER if r < 90 else GREEN for r in sev['ComplianceRate'].values]
bars = ax2.barh(sev_order, sev['ComplianceRate'].values, color=colours)
ax2.axvline(x=90, color=GREEN, linestyle='--', linewidth=1.5, label='Target: 90%')
ax2.axvline(x=75, color=AMBER, linestyle='--', linewidth=1.5, label='Minimum: 75%')
for i, v in enumerate(sev['ComplianceRate'].values):
    ax2.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10)
ax2.set_title('Compliance Rate by Severity')
ax2.set_xlabel('Compliance Rate (%)')
ax2.set_xlim(0, 105)
ax2.legend()

plt.tight_layout()
plt.savefig('chart_01_severity_compliance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')


In [ ]:
# ── 3.4 Breach type analysis ──────────────────────────────────────────────
btype = breaches.groupby('BreachType').agg(
    Total=('BreachID','count'),
    OnTime=('Within72Hours','sum'),
    TotalSubjects=('SubjectsAffected','sum'),
    CriticalCount=('Severity', lambda x: (x=='Critical').sum())
).sort_values('Total', ascending=False)
btype['LateCount'] = btype['Total'] - btype['OnTime']
btype['ComplianceRate'] = (btype['OnTime'] / btype['Total'] * 100).round(1)
print(btype.to_string())


In [ ]:
# ── 3.5 Breach type chart ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
btype_sorted = btype.sort_values('Total')
ax.barh(btype_sorted.index, btype_sorted['Total'], color=RED)
for i, v in enumerate(btype_sorted['Total']):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=10)
ax.set_title('Breach Volume by Type')
ax.set_xlabel('Total Breaches')
plt.tight_layout()
plt.savefig('chart_02_breach_types.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.6 Department risk analysis ──────────────────────────────────────────
dept = breaches.groupby('Department').agg(
    Total=('BreachID','count'),
    Critical=('Severity', lambda x: (x=='Critical').sum()),
    TotalSubjects=('SubjectsAffected','sum'),
    TotalFines=('FineAmountEUR','sum'),
    AvgNotifHours=('NotificationHours','mean')
).sort_values('Total', ascending=False)
dept['CriticalRate'] = (dept['Critical'] / dept['Total'] * 100).round(1)
print(dept.to_string())


In [ ]:
# ── 3.7 Monthly trend ─────────────────────────────────────────────────────
monthly = breaches.groupby(['Year','Month']).agg(
    Total=('BreachID','count'),
    ComplianceRate=('Within72Hours','mean')
).reset_index()
monthly['ComplianceRate'] = monthly['ComplianceRate'] * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
months = range(1, 13)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
colours_yr = {2022: '#90CAF9', 2023: '#5C6BC0', 2024: NAVY}

for yr in [2022, 2023, 2024]:
    d = monthly[monthly['Year']==yr].set_index('Month')
    axes[0].plot(d.index, d['Total'], marker='o', label=str(yr),
                 color=colours_yr[yr], linewidth=2)
    axes[1].plot(d.index, d['ComplianceRate'], marker='o', label=str(yr),
                 color=colours_yr[yr], linewidth=2)

axes[0].set_title('Monthly Breach Volume by Year')
axes[0].set_ylabel('Total Breaches')
axes[0].legend()
axes[0].set_xticks(list(months))
axes[0].set_xticklabels(month_labels)

axes[1].set_title('Monthly 72hr Compliance Rate by Year')
axes[1].set_ylabel('Compliance Rate (%)')
axes[1].axhline(y=90, color=GREEN, linestyle='--', linewidth=1.5, label='Target 90%')
axes[1].axhline(y=75, color=AMBER, linestyle='--', linewidth=1.5, label='Min 75%')
axes[1].set_ylim(0, 110)
axes[1].legend()
axes[1].set_xticks(list(months))
axes[1].set_xticklabels(month_labels)

plt.tight_layout()
plt.savefig('chart_03_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Subject Rights Requests Analysis (Article 12 GDPR)

**Legal context:** Article 12 GDPR requires organisations to respond to data subject rights requests **without undue delay and within one month** (30 days) of receipt. This deadline can be extended by two further months for complex or numerous requests, but the data subject must be informed within the first month.

**Questions this section answers:**
- What is the overall 30-day compliance rate?
- Which request types are most frequent and hardest to fulfil?
- Which departments struggle most with response times?
- What are the most common outcomes — and are refusals legally justified?


In [ ]:
# ── 4.1 Overall DSAR metrics ──────────────────────────────────────────────
completed = requests[requests['Completed']==1]
total_r = len(requests)
comp_r  = len(completed)
on_time_r = requests['Within30Days'].sum()
rate_r = on_time_r / comp_r * 100

print(f'Total Requests:          {total_r}')
print(f'Completed:               {comp_r} ({comp_r/total_r*100:.1f}%)')
print(f'Pending:                 {total_r-comp_r} ({(total_r-comp_r)/total_r*100:.1f}%)')
print(f'Responded Within 30 Days:{int(on_time_r)} ({rate_r:.1f}%)')
print(f'Average Response Days:   {completed["ResponseDays"].mean():.1f}')
print(f'Max Response Days:       {completed["ResponseDays"].max():.0f}')
print()
print('ASSESSMENT:', end=' ')
if rate_r >= 95: print('COMPLIANT (>=95%)')
elif rate_r >= 80: print('AT RISK (80-94%)')
else: print('NON-COMPLIANT (<80%)')


In [ ]:
# ── 4.2 By request type ───────────────────────────────────────────────────
rtype = requests.groupby('RequestType').agg(
    Total=('RequestID','count'),
    Completed=('Completed','sum'),
    OnTime=('Within30Days','sum'),
    AvgDays=('ResponseDays','mean')
).sort_values('Total', ascending=False)
rtype['ComplianceRate'] = (rtype['OnTime'] / rtype['Completed'] * 100).round(1)
rtype['AvgDays'] = rtype['AvgDays'].round(1)
print(rtype.to_string())


In [ ]:
# ── 4.3 Request type chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rtype_sorted = rtype.sort_values('Total')
axes[0].barh(rtype_sorted.index, rtype_sorted['Total'], color=NAVY)
for i, v in enumerate(rtype_sorted['Total']):
    axes[0].text(v+0.3, i, str(v), va='center', fontsize=10)
axes[0].set_title('Requests by Type')
axes[0].set_xlabel('Total Requests')

colours_r = [RED if r < 80 else AMBER if r < 95 else GREEN
             for r in rtype_sorted['ComplianceRate'].values]
axes[1].barh(rtype_sorted.index, rtype_sorted['ComplianceRate'], color=colours_r)
axes[1].axvline(x=95, color=GREEN, linestyle='--', linewidth=1.5, label='Target: 95%')
for i, v in enumerate(rtype_sorted['ComplianceRate'].values):
    axes[1].text(v+0.3, i, f'{v:.1f}%', va='center', fontsize=10)
axes[1].set_title('30-Day Compliance Rate by Type')
axes[1].set_xlabel('Compliance Rate (%)')
axes[1].set_xlim(0, 110)
axes[1].legend()

plt.tight_layout()
plt.savefig('chart_04_dsar_types.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.4 Department compliance ─────────────────────────────────────────────
rdept = requests.groupby('Department').agg(
    Total=('RequestID','count'),
    Completed=('Completed','sum'),
    OnTime=('Within30Days','sum'),
    AvgDays=('ResponseDays','mean')
).sort_values('OnTime')
rdept['ComplianceRate'] = (rdept['OnTime'] / rdept['Completed'] * 100).round(1)
rdept['AvgDays'] = rdept['AvgDays'].round(1)
print(rdept.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
colours_d = [RED if r < 80 else AMBER if r < 95 else GREEN
             for r in rdept['ComplianceRate'].values]
ax.barh(rdept.index, rdept['ComplianceRate'], color=colours_d)
ax.axvline(x=95, color=GREEN, linestyle='--', linewidth=1.5, label='Target: 95%')
ax.axvline(x=80, color=AMBER, linestyle='--', linewidth=1.5, label='Min: 80%')
for i, v in enumerate(rdept['ComplianceRate'].values):
    ax.text(v+0.3, i, f'{v:.1f}%', va='center', fontsize=10)
ax.set_title('30-Day DSAR Compliance by Department')
ax.set_xlabel('Compliance Rate (%)')
ax.set_xlim(0, 110)
ax.legend()
plt.tight_layout()
plt.savefig('chart_05_dsar_dept.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.5 Outcomes analysis ─────────────────────────────────────────────────
outcomes = requests[requests['Completed']==1]['Outcome'].value_counts()
print('Request Outcomes:')
for outcome, count in outcomes.items():
    print(f'  {outcome:<40} {count:>4} ({count/len(completed)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(8, 5))
colours_o = [GREEN, AMBER, NAVY, '#5C6BC0', RED]
wedges, texts, autotexts = ax.pie(
    outcomes.values, labels=outcomes.index,
    autopct='%1.1f%%', colors=colours_o[:len(outcomes)],
    startangle=90, pctdistance=0.75
)
ax.set_title('DSAR Outcomes Distribution')
plt.tight_layout()
plt.savefig('chart_06_dsar_outcomes.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Fine Exposure & Regulatory Actions

**Legal context:** Article 83 GDPR sets out two tiers of administrative fines:
- **Tier 1:** Up to €10M or 2% of global annual turnover — for less severe infringements (Arts. 8, 11, 25-39, 42, 43)
- **Tier 2:** Up to €20M or 4% of global annual turnover — for the most serious infringements (Arts. 5, 6, 7, 9, 12-22)

Supervisory authorities must consider factors including: nature/gravity/duration of infringement, intentional/negligent character, actions taken to mitigate damage, and degree of cooperation.

**Questions this section answers:**
- What is the total fine exposure and how has it been reduced through appeals?
- Which supervisory authorities are most active?
- Which GDPR articles are most frequently breached?
- Is it worth appealing fines?


In [ ]:
# ── 5.1 Overall fine metrics ──────────────────────────────────────────────
total_issued = actions['FineAmountEUR'].sum()
total_final  = actions['FinalAmountEUR'].sum()
reduction    = total_issued - total_final
appealed     = actions['Appealed'].sum()
appeal_rate  = appealed / len(actions) * 100

print(f'Total Fines Issued:      €{total_issued/1e6:.2f}M')
print(f'Total Fines Final:       €{total_final/1e6:.2f}M')
print(f'Total Reduction:         €{reduction/1e6:.2f}M ({reduction/total_issued*100:.1f}%)')
print(f'Total Actions:           {len(actions)}')
print(f'Appeals Lodged:          {int(appealed)} ({appeal_rate:.1f}%)')
print(f'Average Fine:            €{actions["FineAmountEUR"].mean()/1e6:.2f}M')
print(f'Largest Fine:            €{actions["FineAmountEUR"].max()/1e6:.2f}M')


In [ ]:
# ── 5.2 Appeal outcomes analysis ──────────────────────────────────────────
appealed_df = actions[actions['Appealed']==1]
outcomes_a = appealed_df.groupby('AppealOutcome').agg(
    Count=('ActionID','count'),
    OriginalFine=('FineAmountEUR','sum'),
    FinalFine=('FinalAmountEUR','sum')
)
outcomes_a['Reduction'] = outcomes_a['OriginalFine'] - outcomes_a['FinalFine']
outcomes_a['ReductionPct'] = (outcomes_a['Reduction']/outcomes_a['OriginalFine']*100).round(1)
print('Appeal Outcomes:')
print(outcomes_a.to_string())
print()
print(f'SUCCESS: Appeals reduced total exposure by €{outcomes_a["Reduction"].sum()/1e6:.2f}M')
print(f'Reduced outcome rate: {len(appealed_df[appealed_df["AppealOutcome"]=="Reduced"])/len(appealed_df)*100:.1f}%')


In [ ]:
# ── 5.3 By supervisory authority ──────────────────────────────────────────
auth = actions.groupby('SupervisoryAuth').agg(
    Actions=('ActionID','count'),
    TotalFines=('FineAmountEUR','sum'),
    AvgFine=('FineAmountEUR','mean'),
    Appeals=('Appealed','sum')
).sort_values('TotalFines', ascending=False)
auth['AppealRate'] = (auth['Appeals']/auth['Actions']*100).round(1)
auth['TotalFinesM'] = (auth['TotalFines']/1e6).round(2)
auth['AvgFineM'] = (auth['AvgFine']/1e6).round(2)
print(auth[['Actions','TotalFinesM','AvgFineM','Appeals','AppealRate']].to_string())

fig, ax = plt.subplots(figsize=(12, 5))
auth_sorted = auth.sort_values('TotalFines')
ax.barh(auth_sorted.index, auth_sorted['TotalFines']/1e6, color=RED)
for i, v in enumerate(auth_sorted['TotalFines'].values/1e6):
    ax.text(v+1, i, f'€{v:.0f}M', va='center', fontsize=10)
ax.set_title('Total Fines by Supervisory Authority')
ax.set_xlabel('Total Fines (€M)')
plt.tight_layout()
plt.savefig('chart_07_authority_fines.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.4 Most breached articles ────────────────────────────────────────────
articles = actions.groupby('ArticleBreached').agg(
    Count=('ActionID','count'),
    TotalFines=('FineAmountEUR','sum'),
    AvgFine=('FineAmountEUR','mean')
).sort_values('TotalFines', ascending=False)
articles['TotalFinesM'] = (articles['TotalFines']/1e6).round(2)
articles['AvgFineM'] = (articles['AvgFine']/1e6).round(2)
print(articles[['Count','TotalFinesM','AvgFineM']].to_string())


## 6. Audit Findings & Risk Assessment

**Legal context:** While GDPR does not mandate internal audits, Article 24 requires controllers to implement appropriate technical and organisational measures. Internal audits are a key accountability mechanism under Article 5(2). Recurring findings in a regulatory audit context can be treated as **aggravating factors** when calculating fines under Article 83(2)(d).

**Questions this section answers:**
- What is our overall remediation rate and backlog?
- Which finding types are most frequent and most persistent (recurring)?
- Which departments carry the highest unresolved risk?
- Is the remediation backlog growing or shrinking over time?


In [ ]:
# ── 6.1 Overall audit metrics ─────────────────────────────────────────────
total_f = len(findings)
remediated = findings['Remediated'].sum()
outstanding = total_f - remediated
recurring = findings['Recurring'].sum()
rem_rate = remediated / total_f * 100
rec_rate = recurring / total_f * 100

print(f'Total Findings:          {total_f}')
print(f'Remediated:              {int(remediated)} ({rem_rate:.1f}%)')
print(f'Outstanding:             {int(outstanding)} ({100-rem_rate:.1f}%)')
print(f'Recurring Findings:      {int(recurring)} ({rec_rate:.1f}%)')
print(f'Avg Remediation Days:    {findings[findings["Remediated"]==1]["RemediationDays"].mean():.1f}')
print()
print('REMEDIATION STATUS:', end=' ')
if rem_rate >= 80: print('ON TRACK (>=80%)')
elif rem_rate >= 60: print('REVIEW NEEDED (60-79%)')
else: print('CRITICAL BACKLOG (<60%)')


In [ ]:
# ── 6.2 By risk level ─────────────────────────────────────────────────────
risk_order = ['Critical','High','Medium','Low']
risk = findings.groupby('RiskLevel').agg(
    Total=('FindingID','count'),
    Remediated=('Remediated','sum'),
    Recurring=('Recurring','sum'),
    AvgDays=('RemediationDays','mean')
).reindex(risk_order)
risk['Outstanding'] = risk['Total'] - risk['Remediated']
risk['RemediationRate'] = (risk['Remediated']/risk['Total']*100).round(1)
risk['RecurringRate'] = (risk['Recurring']/risk['Total']*100).round(1)
risk['AvgDays'] = risk['AvgDays'].round(1)
print(risk.to_string())


In [ ]:
# ── 6.3 Risk level chart ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

risk_colours = {'Critical': RED, 'High': AMBER, 'Medium': '#F9A825', 'Low': GREEN}
bar_colours = [risk_colours[r] for r in risk_order]

# Stacked bar
axes[0].barh(risk_order, risk['Remediated'], color=GREEN, label='Remediated')
axes[0].barh(risk_order, risk['Outstanding'], left=risk['Remediated'], color=RED, label='Outstanding')
for i, (r, o) in enumerate(zip(risk['Remediated'], risk['Outstanding'])):
    axes[0].text(r/2, i, str(int(r)), ha='center', va='center', color='white', fontweight='bold')
    axes[0].text(r+o/2, i, str(int(o)), ha='center', va='center', color='white', fontweight='bold')
axes[0].set_title('Remediated vs Outstanding by Risk Level')
axes[0].legend()

# Remediation rate
axes[1].barh(risk_order, risk['RemediationRate'], color=bar_colours)
axes[1].axvline(x=80, color=GREEN, linestyle='--', linewidth=1.5, label='Target: 80%')
for i, v in enumerate(risk['RemediationRate'].values):
    axes[1].text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=10)
axes[1].set_title('Remediation Rate by Risk Level')
axes[1].set_xlabel('Remediation Rate (%)')
axes[1].set_xlim(0, 110)
axes[1].legend()

plt.tight_layout()
plt.savefig('chart_08_audit_risk.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 6.4 Finding types ─────────────────────────────────────────────────────
ftype = findings.groupby('FindingType').agg(
    Total=('FindingID','count'),
    Remediated=('Remediated','sum'),
    Recurring=('Recurring','sum')
).sort_values('Total', ascending=False)
ftype['RemediationRate'] = (ftype['Remediated']/ftype['Total']*100).round(1)
ftype['RecurringRate'] = (ftype['Recurring']/ftype['Total']*100).round(1)
print(ftype.to_string())


## 7. Processing Activities Register (Article 30 GDPR)

**Legal context:** Article 30 GDPR requires controllers to maintain a **record of processing activities** containing information on purposes, data categories, recipients, retention periods, and security measures. Article 35 requires a **Data Protection Impact Assessment (DPIA)** for processing likely to result in high risk to individuals — particularly for systematic processing of special categories of data.

**Questions this section answers:**
- What percentage of processing activities are compliant?
- Is the lawful basis correctly applied across activity types?
- Which activities require a DPIA that hasn't been completed?
- What cross-border transfer mechanisms are in use post-Schrems II?


In [ ]:
# ── 7.1 Overall compliance ────────────────────────────────────────────────
total_a = len(activities)
compliant = activities['Compliant'].sum()
dpia_req = activities['DPIARequired'].sum()
dpia_done = activities['DPIACompleted'].sum()
cross_border = activities['CrossBorderTransfer'].sum()
third_party = activities['ThirdPartySharing'].sum()

print(f'Total Activities:        {total_a}')
print(f'Compliant:               {int(compliant)} ({compliant/total_a*100:.1f}%)')
print(f'Non-Compliant:           {int(total_a-compliant)} ({(total_a-compliant)/total_a*100:.1f}%)')
print(f'DPIA Required:           {int(dpia_req)} ({dpia_req/total_a*100:.1f}%)')
print(f'DPIA Completed:          {int(dpia_done)} ({dpia_done/dpia_req*100:.1f}% of required)')
print(f'DPIA Outstanding:        {int(dpia_req-dpia_done)}')
print(f'Cross-Border Transfers:  {int(cross_border)} ({cross_border/total_a*100:.1f}%)')
print(f'Third Party Sharing:     {int(third_party)} ({third_party/total_a*100:.1f}%)')


In [ ]:
# ── 7.2 Lawful basis analysis ─────────────────────────────────────────────
basis = activities.groupby('LawfulBasis').agg(
    Total=('ActivityID','count'),
    Compliant=('Compliant','sum'),
    DPIARequired=('DPIARequired','sum'),
    CrossBorder=('CrossBorderTransfer','sum')
).sort_values('Total', ascending=False)
basis['ComplianceRate'] = (basis['Compliant']/basis['Total']*100).round(1)
print(basis.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
basis_sorted = basis.sort_values('Total')
axes[0].barh(basis_sorted.index, basis_sorted['Total'], color=NAVY)
axes[0].set_title('Activities by Lawful Basis (Art. 6)')
axes[0].set_xlabel('Total Activities')
for i, v in enumerate(basis_sorted['Total']):
    axes[0].text(v+0.1, i, str(v), va='center')

cr_colours = [RED if r < 75 else AMBER if r < 90 else GREEN
              for r in basis_sorted['ComplianceRate'].values]
axes[1].barh(basis_sorted.index, basis_sorted['ComplianceRate'], color=cr_colours)
axes[1].axvline(x=90, color=GREEN, linestyle='--', linewidth=1.5, label='Target: 90%')
axes[1].set_title('Compliance Rate by Lawful Basis')
axes[1].set_xlabel('Compliance Rate (%)')
axes[1].set_xlim(0, 110)
axes[1].legend()
for i, v in enumerate(basis_sorted['ComplianceRate'].values):
    axes[1].text(v+0.3, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.savefig('chart_09_lawful_basis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.3 Transfer mechanism analysis ───────────────────────────────────────
cross = activities[activities['CrossBorderTransfer']==1]
mechanisms = cross.groupby('TransferMechanism').agg(
    Total=('ActivityID','count'),
    Compliant=('Compliant','sum')
)
mechanisms['ComplianceRate'] = (mechanisms['Compliant']/mechanisms['Total']*100).round(1)
mechanisms['Share'] = (mechanisms['Total']/len(cross)*100).round(1)
print('Cross-Border Transfer Mechanisms (post-Schrems II):')
print(mechanisms.to_string())
print()
print('NOTE: Standard Contractual Clauses (SCCs) are the primary')
print('transfer mechanism following the CJEU Schrems II ruling (C-311/18)')
print('which invalidated Privacy Shield in July 2020.')


In [ ]:
# ── 7.4 Special category data analysis ────────────────────────────────────
special_categories = ['Health Data', 'Criminal Records', 'Biometric Data']
special = activities[activities['DataCategory'].isin(special_categories)]
print(f'Special Category Data Activities: {len(special)} of {total_a} ({len(special)/total_a*100:.1f}%)')
print()
print('By Category:')
for cat in special_categories:
    n = len(activities[activities['DataCategory']==cat])
    comp = activities[activities['DataCategory']==cat]['Compliant'].sum()
    print(f'  {cat:<25} {n:>3} activities | {comp/n*100:.1f}% compliant')
print()
print('NOTE: Special category data (Art. 9) requires explicit consent')
print('or specific exemptions. Higher compliance scrutiny applies.')


## 8. Key Findings Summary

In [ ]:
print('='*60)
print('GDPR EDA — KEY FINDINGS SUMMARY')
print('='*60)

print(f'''
BREACH COMPLIANCE (Art. 33):
  Overall 72hr Rate:     {breaches["Within72Hours"].mean()*100:.1f}%  [TARGET: 90%]
  Status:                NON-COMPLIANT
  Avg Notification:      {breaches["NotificationHours"].mean():.1f} hours (limit: 72)
  Most Affected:         Low severity — {breaches[breaches["Severity"]=="Low"].shape[0]} breaches, 
                         117 notified late
  Critical compliance:   {breaches[breaches["Severity"]=="Critical"]["Within72Hours"].mean()*100:.1f}%

DSAR COMPLIANCE (Art. 12):
  30-Day Rate:           {requests["Within30Days"].sum()/requests["Completed"].sum()*100:.1f}%  [TARGET: 95%]
  Status:                NON-COMPLIANT
  Avg Response:          {requests["ResponseDays"].mean():.1f} days (limit: 30)
  Pending Requests:      {(requests["Completed"]==0).sum()}

FINE EXPOSURE:
  Total Issued:          €{actions["FineAmountEUR"].sum()/1e6:.2f}M
  After Appeals:         €{actions["FinalAmountEUR"].sum()/1e6:.2f}M
  Total Reduced:         €{(actions["FineAmountEUR"].sum()-actions["FinalAmountEUR"].sum())/1e6:.2f}M
  Appeal Success Rate:   {len(actions[(actions["Appealed"]==1)&(actions["AppealOutcome"]=="Reduced")])/actions["Appealed"].sum()*100:.1f}%

AUDIT FINDINGS:
  Remediation Rate:      {findings["Remediated"].mean()*100:.1f}%  [TARGET: 80%]
  Status:                REVIEW NEEDED
  Outstanding:           {int((findings["Remediated"]==0).sum())} findings
  Recurring Rate:        {findings["Recurring"].mean()*100:.1f}%

PROCESSING ACTIVITIES:
  Compliance Rate:       {activities["Compliant"].mean()*100:.1f}%  [TARGET: 90%]
  DPIA Completion:       {activities["DPIACompleted"].sum()/activities["DPIARequired"].sum()*100:.1f}%
  DPIA Outstanding:      {int(activities["DPIARequired"].sum()-activities["DPIACompleted"].sum())}
  Cross-Border:          {int(activities["CrossBorderTransfer"].sum())} activities
''')
print('='*60)
